# Clase 2: Optimizacion y Entrenamiento de Redes Neuronales

## Caso de Estudio: Prediccion de Supervivencia con el Titanic Dataset

**Modulo:** Cientifico de Datos e Inteligencia Artificial Aplicada
**Tecnologias:** Python, TensorFlow, Keras, Scikit-Learn, Matplotlib
**Dataset:** [Titanic - Machine Learning from Disaster (Kaggle)](https://www.kaggle.com/competitions/titanic)

## Objetivo de esta practica

En la Clase 1 construimos una ANN que funcionaba. Ahora vamos a **entrenarla bien**.

Partimos de un modelo baseline y medimos, con evidencia, el impacto de cada decision de entrenamiento:
optimizador -> learning rate -> batch size -> inicializacion -> regularizacion -> callbacks.

Flujo: datos de Kaggle -> baseline -> experimentos comparativos -> modelo final optimizado -> evaluacion.

## 0) Obtener el dataset desde Kaggle

Los datasets **no se versionan en el repositorio**. Cada estudiante los descarga en su entorno.

### Opcion A: descarga manual
1. Crear cuenta en https://www.kaggle.com
2. Entrar a https://www.kaggle.com/competitions/titanic y aceptar las reglas (`Join Competition`).
3. Pestana `Data` -> descargar `train.csv`.
4. Guardarlo en la carpeta `data/` de esta clase: `data/train.csv`.

### Opcion B: API oficial de Kaggle
```bash
pip install kaggle
# Token: https://www.kaggle.com/settings -> API -> Create New Token
# Windows: C:\Users\<usuario>\.kaggle\kaggle.json
# Linux/macOS: ~/.kaggle/kaggle.json   (chmod 600 ~/.kaggle/kaggle.json)
kaggle competitions download -c titanic -p data
```
Luego descomprimir `data/titanic.zip`.

> Recuerda anadir `data/` y `kaggle.json` al `.gitignore`.

### Lectura del EDA

- `Age` y `Embarked` tienen nulos: hay que imputar, no eliminar filas (perderiamos ~20% del dataset).
- `Cabin` tiene demasiados nulos para ser util en este baseline.
- `Name`, `Ticket` y `PassengerId` son identificadores: se descartan para evitar ruido y leakage.
- El target esta moderadamente desbalanceado (~62% / 38%), aceptable sin `class_weight`.

> **Importante:** el preprocesador se ajusta (`fit`) solo con `X_train`. Si se ajustara con todo el
> dataset, la media y la desviacion de test se filtrarian al entrenamiento (**data leakage**).

## 5) Funciones de apoyo para experimentar

Para comparar decisiones de entrenamiento de forma justa necesitamos que **todo lo demas se mantenga igual**:
misma arquitectura, misma semilla, mismas epocas. Estas dos funciones nos dan esa base controlada.

## 7) Descenso de gradiente y optimizadores

El entrenamiento actualiza los pesos siguiendo el gradiente de la perdida:

$$
w \leftarrow w - \eta \frac{\partial L}{\partial w}
$$

**Backpropagation** es el algoritmo que calcula esos gradientes hacia atras, capa por capa.
El **optimizador** decide como se usa el gradiente:

| Optimizador | Idea central |
|---|---|
| `SGD` | Aplica la regla base tal cual |
| `SGD + Momentum` | Acumula la direccion previa: atraviesa zonas planas |
| `RMSprop` | Learning rate adaptativo por parametro |
| `Adam` | Momentum + RMSprop: el default razonable |

### Como leer esta comparativa

- **SGD puro** baja la loss de forma lenta y suave: necesita mas epocas o un learning rate mayor.
- **Momentum** acelera claramente a SGD con el mismo learning rate.
- **RMSprop y Adam** convergen rapido en las primeras epocas, pero suelen empezar a sobreajustar antes.
- La curva que importa para decidir es la de **validacion**, no la de train.

## 8) Learning rate: el hiperparametro mas critico

| Learning rate | Sintoma |
|---|---|
| Muy alto | La loss oscila, sube o se vuelve `NaN` |
| Muy bajo | La loss baja demasiado lento, no llega a converger |
| Adecuado | Baja rapido al inicio y luego se estabiliza |

## 9) Schedulers: learning rate que cambia en el tiempo

La estrategia es dar **pasos grandes al inicio** para avanzar rapido y **pasos pequenos al final**
para afinar sin saltarse el minimo.

- `ExponentialDecay`: decae de forma continua segun los pasos ejecutados.
- `PiecewiseConstantDecay`: escalones definidos manualmente.
- `ReduceLROnPlateau`: reduce el lr **solo** cuando `val_loss` deja de mejorar (callback reactivo).

## 10) Batch size

Define cuantas muestras se procesan antes de cada actualizacion de pesos.

| Batch size | Efecto |
|---|---|
| Pequeno (8-32) | Mas ruido en el gradiente, a veces mejor generalizacion, mas lento por epoca |
| Mediano (32-128) | Equilibrio habitual |
| Grande (256+) | Rapido y estable, riesgo de converger a minimos que generalizan peor |

## 11) Inicializacion de pesos

Si todos los pesos inician en el mismo valor, todas las neuronas calculan lo mismo y la red nunca
se diferencia (problema de simetria). Una buena inicializacion mantiene la varianza de la senal
entre capas.

| Inicializador | Recomendado con |
|---|---|
| `glorot_uniform` (Xavier) | `tanh`, `sigmoid` — es el default de Keras |
| `he_normal` | `relu` y variantes |
| `zeros` | **Nunca**: rompe el entrenamiento (se incluye solo para demostrarlo) |

## 12) Regularizacion: controlar el overfitting

Forzamos primero el overfitting con una red sobredimensionada y luego aplicamos cada tecnica.

### L2 (weight decay)
$$
L_{total} = L + \lambda \sum w^2
$$
Penaliza pesos grandes. L1 (`\lambda \sum |w|`) tiende a llevar pesos a cero.

### Dropout
Desactiva aleatoriamente un porcentaje de neuronas en cada paso de entrenamiento (valores tipicos 0.2-0.5).
En inferencia se desactiva solo.

### Batch Normalization
Normaliza las activaciones por mini-batch: estabiliza el entrenamiento y permite learning rates mas altos.

### Senal de overfitting

En la red grande sin regularizar la `loss` de train sigue bajando mientras la `val_loss` empieza a
**subir**: el modelo esta memorizando el conjunto de entrenamiento. Las variantes regularizadas
mantienen la brecha entre train y validacion mucho mas estrecha.

## 13) Callbacks: no elegir las epocas a mano

| Callback | Funcion |
|---|---|
| `EarlyStopping` | Detiene el entrenamiento cuando `val_loss` deja de mejorar |
| `ModelCheckpoint` | Guarda el mejor modelo segun la metrica de validacion |
| `ReduceLROnPlateau` | Reduce el learning rate cuando la mejora se estanca |

La practica correcta es entrenar con un limite alto de epocas y dejar que `EarlyStopping`
con `restore_best_weights=True` recupere el mejor punto.

## Como leer estas graficas (diagnostico)

| Situacion | Train | Validacion | Diagnostico | Accion |
|---|---|---|---|---|
| Ambas bajan y quedan cercanas | Loss baja | Loss baja | Buen ajuste | Consolidar |
| Train baja, validacion sube | Loss baja | Loss sube | **Overfitting** | Dropout, L2, EarlyStopping, mas datos |
| Ambas se quedan altas | Loss alta | Loss alta | **Underfitting** | Mas capas/neuronas, mas epocas, mayor lr |
| La loss oscila fuerte | Inestable | Inestable | Learning rate alto | Reducir lr o usar scheduler |
| Validacion mejor que train | - | - | Dropout activo en train | Normal, confirmar en test |

### Matriz de confusion (Test)
- Filas: clase real. Columnas: clase predicha.
- Diagonal principal: aciertos. Fuera de la diagonal: errores.
- En Titanic interesa mirar los **falsos negativos** (pasajeros que sobrevivieron y el modelo
  clasifico como no sobrevivientes): si el costo de ese error es alto, conviene bajar el umbral de 0.5.

### Linea del punto rojo en la curva de Loss
Marca la epoca con `val_loss` minima, que es el modelo que `EarlyStopping` restauro con
`restore_best_weights=True`. Las epocas posteriores solo estaban memorizando.

## Actividad rapida

1. Cambia el optimizador del modelo final a `SGD(learning_rate=0.01, momentum=0.9)` y compara el
   accuracy en test. Documenta cual gano y por que crees que fue asi.
2. Ajusta el umbral de decision (`0.3`, `0.4`, `0.6`) y analiza como cambian precision y recall
   de la clase `Sobrevivio`.
3. Reduce `patience` de `EarlyStopping` a 3 y explica que riesgo introduce detenerse demasiado pronto.
4. Prueba `dropout=0.6` y justifica con las curvas si ayuda o si provoca underfitting.
5. Crea una variable nueva (por ejemplo `FamilySize = SibSp + Parch + 1`), reentrena el modelo final
   y reporta si el aporte de la variable supera el de los ajustes de optimizacion.

## Conclusion de la clase

La arquitectura define **que puede aprender** la red; la optimizacion define **si realmente lo aprende**.
Un baseline con buenas decisiones de entrenamiento supera a una red grande mal entrenada.